# Hurricane Database 1851–2024

Starter notebook for the Kaggle dataset in this folder. It loads `hurricanes.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `hurricanes.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("hurricanes.csv", low_memory=False)
df["record_time_utc"] = pd.to_datetime(df["record_time_utc"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Storms and waypoints by decade

In [ ]:
df.groupby("decade").agg(storms=("storm_id", "nunique"),
                              waypoints=("storm_id", "size")).tail(12)

## 5. Peak category distribution

In [ ]:
(df.drop_duplicates("storm_id")["storm_peak_category"]
   .value_counts().reindex(["Tropical Depression", "Tropical Storm", "Category 1", "Category 2",
                            "Category 3", "Category 4", "Category 5"]))

## 6. Accumulated Cyclone Energy by decade

In [ ]:
ace = df.drop_duplicates("storm_id").groupby("decade")["storm_ace"].sum()
ace.tail(15).plot(kind="bar", figsize=(10, 4), title="Total ACE by decade")

## 7. A single track, mapped

In [ ]:
storm = df[df["storm_id"] == df.loc[df["storm_max_wind_kt"].idxmax(), "storm_id"]]
ax = storm.plot(x="longitude", y="latitude", figsize=(8, 6), legend=False,
                title=f"Track of {storm['storm_name'].iloc[0]} ({storm['storm_year'].iloc[0]})")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

## 8. Wind vs pressure

In [ ]:
sub = df.dropna(subset=["min_pressure_mb", "max_wind_kt"])
sub.plot.scatter(x="min_pressure_mb", y="max_wind_kt", s=4, alpha=0.2, figsize=(8, 5))

## 9. Translation speed over time

In [ ]:
df.groupby("decade")["movement_speed_kmh"].median().tail(15).plot(
    kind="line", marker="o", figsize=(10, 4), title="Median translation speed by decade")

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.